# Module 3.2: Decision Trees for Interpretable Machine Fault Diagnosis
### 🇵🇱 Drzewa Decyzyjne w Interpretowalnej Diagnostyce Usterek Maszyn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m03_l02_classification_decision_trees_faults.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Train white-box decision trees on engineered vibration features, visualize decision boundaries, and export logic directly to PLC ladder rules.  
> 🇵🇱 **Streszczenie:** Trenuj interpretowalne drzewa decyzyjne na cechach drganiowych, wizualizuj granice podziału i eksportuj reguły do logiki drabinkowej PLC.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Discrete Fault Diagnostics**: Classifying operational states (Normal, Cavitation, Bearing Defect, Line Blockage).
2. **Interpretability in Regulated Plants**: Black-box models cannot be deployed in safety-critical manufacturing. Reliability engineers demand human-auditable logic.
3. **PLC Ladder Logic / Structured Text Export**: A trained `DecisionTreeClassifier` can be compiled into IEC 61131-3 `IF-THEN` statements for execution on Siemens S7-1500 or Allen-Bradley PLCs.


In [ ]:
# 1. Synthesize Pump Fault Diagnostic Dataset
from sklearn.tree import DecisionTreeClassifier, export_text

np.random.seed(42)
n_records = 1200

# Features: Discharge Pressure (bar), Motor Current (Amps), Vibration RMS (m/s²)
# State 0: Normal
# State 1: Cavitation (Low pressure, surging vibration)
# State 2: Bearing Outer Race Fault (High vibration, normal pressure)
n_per_class = 400

f_normal = np.random.multivariate_normal([160.0, 25.0, 1.2], np.diag([4, 2, 0.05]), n_per_class)
f_cavit  = np.random.multivariate_normal([110.0, 28.0, 4.5], np.diag([9, 3, 0.40]), n_per_class)
f_bear   = np.random.multivariate_normal([158.0, 31.0, 6.8], np.diag([5, 2, 0.60]), n_per_class)

X_pump = np.vstack([f_normal, f_cavit, f_bear])
y_pump = np.array([0]*n_per_class + [1]*n_per_class + [2]*n_per_class)
feature_names = ["pressure_bar", "motor_current_a", "vibration_rms"]
target_names = ["NORMAL", "CAVITATION", "BEARING_FAULT"]

print(f"✓ Generated {len(X_pump)} diagnostic records across 3 fault modes.")


In [ ]:
# 2. Train Interpretable Decision Tree & Export IEC 61131-3 PLC Code
clf_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_tree.fit(X_pump, y_pump)

print("--- Decision Tree Text Representation ---")
print(export_text(clf_tree, feature_names=feature_names))

def export_to_plc_structured_text(tree, features, classes):
    """Converts scikit-learn DecisionTree into IEC 61131-3 Structured Text for PLC"""
    left = tree.tree_.children_left
    right = tree.tree_.children_right
    threshold = tree.tree_.threshold
    feature = tree.tree_.feature
    value = tree.tree_.value

    lines = ["// --- Auto-Generated IEC 61131-3 Structured Text ---", "CASE eState OF"]
    
    def recurse(node, indent):
        sp = "  " * indent
        if left[node] != right[node]:
            f_name = features[feature[node]]
            thresh = threshold[node]
            lines.append(f"{sp}IF ({f_name} <= {thresh:.2f}) THEN")
            recurse(left[node], indent + 1)
            lines.append(f"{sp}ELSE")
            recurse(right[node], indent + 1)
            lines.append(f"{sp}END_IF;")
        else:
            cls_idx = np.argmax(value[node])
            lines.append(f"{sp}FaultCode := {cls_idx}; // {classes[cls_idx]}")

    recurse(0, 1)
    lines.append("END_CASE;")
    return "\n".join(lines)

plc_code = export_to_plc_structured_text(clf_tree, feature_names, target_names)
print(plc_code)


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Train a DecisionTreeClassifier to categorize machinery states (Healthy, Unbalance, Bearing Fault) *(🇵🇱 Wytrenuj klasyfikator drzewa decyzyjnego do podziału stanów maszyny (Sprawna, Niewyważenie, Awaria Łożyska))*
- [ ] **Task 2:** Tune tree depth (max_depth) and minimum split samples to prevent overfitting *(🇵🇱 Dobierz maksymalną głębokość drzewa i min_samples_split, zapobiegając przeuczeniu)*
- [ ] **Task 3:** Export tree decision path rules into human-readable PLC IF-THEN ladder logic *(🇵🇱 Wyeksportuj reguły drzewa decyzyjnego do czytelnej logiki IF-THEN dla sterownika PLC)*


In [ ]:
# Exercise Challenge:
# Assert that the decision tree accuracy on training telemetry exceeds 98%
train_acc = clf_tree.score(X_pump, y_pump)
print(f"Tree Diagnostic Accuracy: {train_acc*100:.1f}%")
assert train_acc >= 0.98, "Decision tree failed to achieve target accuracy."
print("✓ Decision tree PLC export verified.")
